# Preliminary Results — Mid-Report Figures

**Task:** EN-trained models → zero-shot transfer to Roman Urdu

Run this notebook **after** `scripts/run_ablations.py` and `scripts/few_shot_train.py`.

Expected files in `outputs/predictions/`:
- `mbert_zero_shot_ur_metrics.json`
- `xlmr_base_zero_shot_ur_metrics.json`
- `xlmr_large_zero_shot_ur_metrics.json`
- `few_shot_summary.csv`

In [ ]:
import sys, os, json
os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style='whitegrid', palette='Set2')
os.makedirs('outputs/figures', exist_ok=True)

In [ ]:
# ── Load backbone comparison results ─────────────────────────────────────
BACKBONE_TAGS = {
    'mBERT':        'mbert_zero_shot_ur',
    'XLM-R Base':   'xlmr_base_zero_shot_ur',
    'XLM-R Large':  'xlmr_large_zero_shot_ur',
}

rows = []
for model_label, tag in BACKBONE_TAGS.items():
    path = f'outputs/predictions/{tag}_metrics.json'
    if not os.path.exists(path):
        print(f'Missing: {path} — run scripts/run_ablations.py first')
        continue
    with open(path) as f:
        m = json.load(f)
    rows.append({'Model': model_label,
                 'Hate F1':     round(m['f1_hate'],     4),
                 'Non-hate F1': round(m['f1_non_hate'], 4),
                 'Macro F1':    round(m['f1_macro'],    4),
                 'FPR':         round(m['fpr'],         4),
                 'Accuracy':    round(m['accuracy'],    4)})

results_df = pd.DataFrame(rows)
print('Zero-shot EN→Urdu Results:')
display(results_df)

In [ ]:
# ── Figure: Zero-shot backbone comparison ────────────────────────────────
if not results_df.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(results_df))
    w = 0.25
    metrics = ['Hate F1', 'Non-hate F1', 'Macro F1']
    colors  = ['#e74c3c', '#2ecc71', '#3498db']
    for i, (col, color) in enumerate(zip(metrics, colors)):
        ax.bar(x + (i-1)*w, results_df[col], w, label=col, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(results_df['Model'])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Zero-Shot Cross-Lingual Transfer (EN → Roman Urdu) — Model Comparison')
    ax.legend()
    plt.tight_layout()
    plt.savefig('outputs/figures/zero_shot_backbone_comparison.png', dpi=150)
    plt.show()
    print('Saved → outputs/figures/zero_shot_backbone_comparison.png')

In [ ]:
# ── Few-shot learning curve ────────────────────────────────────────────────
fs_path = 'outputs/predictions/few_shot_summary.csv'

if os.path.exists(fs_path):
    fs_df = pd.read_csv(fs_path)

    # Prepend zero-shot (K=0) row from XLM-R large
    zs_path = 'outputs/predictions/xlmr_large_zero_shot_ur_metrics.json'
    if os.path.exists(zs_path):
        with open(zs_path) as f: zs = json.load(f)
        zs_row = pd.DataFrame([{'k': 0,
                                 'f1_hate_mean': zs['f1_hate'],  'f1_hate_std': 0,
                                 'f1_macro_mean': zs['f1_macro'], 'f1_macro_std': 0,
                                 'fpr_mean': zs['fpr']}])
        fs_df = pd.concat([zs_row, fs_df], ignore_index=True).sort_values('k')

    display(fs_df.round(4))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(fs_df['k'], fs_df['f1_hate_mean'],  'o-',  color='#e74c3c', linewidth=2, label='Hate F1')
    ax.fill_between(fs_df['k'],
                    fs_df['f1_hate_mean'] - fs_df['f1_hate_std'],
                    fs_df['f1_hate_mean'] + fs_df['f1_hate_std'],
                    alpha=0.2, color='#e74c3c')
    ax.plot(fs_df['k'], fs_df['f1_macro_mean'], 's--', color='#3498db', linewidth=2, label='Macro F1')
    ax.set_xlabel('Number of Urdu training examples (K-shot)')
    ax.set_ylabel('F1 Score on Urdu Test Set')
    ax.set_title('Few-Shot Learning Curve — EN-trained XLM-R Large → Roman Urdu')
    ax.legend()
    ax.set_ylim(0, 1.0)
    plt.tight_layout()
    plt.savefig('outputs/figures/few_shot_learning_curve.png', dpi=150)
    plt.show()
    print('Saved → outputs/figures/few_shot_learning_curve.png')
else:
    print('Few-shot summary not found — run scripts/few_shot_train.py first')

In [ ]:
# ── Confusion matrix — best model zero-shot Urdu ─────────────────────────
pred_path = 'outputs/predictions/xlmr_large_zero_shot_ur_predictions.json'

if os.path.exists(pred_path):
    import json
    with open(pred_path) as f: preds = json.load(f)
    preds = [p for p in preds if p['true_label'] != -1]
    true = [p['true_label'] for p in preds]
    pred = [p['predicted_label'] for p in preds]

    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(true, pred, labels=[0,1])
    ConfusionMatrixDisplay(cm, display_labels=['Non-hate', 'Hate']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title('Confusion Matrix — XLM-R Large Zero-Shot (EN→Urdu)')
    plt.tight_layout()
    plt.savefig('outputs/figures/confusion_matrix_xlmr_large_zeroshot_ur.png', dpi=150)
    plt.show()
else:
    print('Prediction file not found — run scripts/zero_shot_eval.py first')